# Explore Madrid Accident Data

This notebook explores the raw CSV data from Madrid's Open Data portal to decide which columns should be stored in the new `accident_participants` table.

In [1]:
import pandas as pd
import io
import requests
from pathlib import Path

# Sample URL for 2024 accidents
URL_2024 = "https://datos.madrid.es/dataset/300228-0-accidentes-trafico-detalle/resource/300228-2-accidentes-trafico-detalle-csv/download/300228-2-accidentes-trafico-detalle-csv.csv"

print("Downloading 2024 data...")
resp = requests.get(URL_2024)
df = pd.read_csv(io.BytesIO(resp.content), sep=";", encoding="utf-8-sig")

# Normalize columns
df.columns = [c.lower().strip().replace(" ", "_") for c in df.columns]

print(f"Total records: {len(df)}")
print(f"Unique accidents: {df['num_expediente'].nunique()}")
df.head()

Total records: 49340
Unique accidents: 20698


,num_expediente,fecha,hora,localizacion,numero,cod_distrito,distrito,tipo_accidente,estado_meteorológico,tipo_vehiculo,tipo_persona,rango_edad,sexo,cod_lesividad,lesividad,coordenada_x_utm,coordenada_y_utm,positiva_alcohol,positiva_droga
0,2023S040280,04/01/2024,14:09:00,AVDA. NICETO ALCALA ZAMORA / AUTOV. M-11,3,16,HORTALEZA,Colisión fronto-lateral,Lluvia débil,Motocicleta > 125cc,Conductor,De 55 a 59 años,Hombre,2.0,Ingreso inferior o igual a 24 horas,444913.0,4481427.0,N,NaN
1,2023S040280,04/01/2024,14:09:00,AVDA. NICETO ALCALA ZAMORA / AUTOV. M-11,3,16,HORTALEZA,Colisión fronto-lateral,Lluvia débil,Turismo,Conductor,De 55 a 59 años,Mujer,14.0,Sin asistencia sanitaria,444913.0,4481427.0,N,NaN
2,2023S040309,15/02/2024,14:05:00,CALL. TESORO / CALL. MINAS,18,1,CENTRO,Colisión fronto-lateral,Lluvia débil,Bicicleta,Conductor,De 25 a 29 años,Hombre,7.0,Asistencia sanitaria sólo en el lugar del acci...,440123.0,4475170.0,N,NaN
3,2023S040309,15/02/2024,14:05:00,CALL. TESORO / CALL. MINAS,18,1,CENTRO,Colisión fronto-lateral,Lluvia débil,Motocicleta hasta 125cc,Conductor,De 35 a 39 años,Hombre,14.0,Sin asistencia sanitaria,440123.0,4475170.0,N,NaN
4,2023S040310,18/02/2024,10:40:00,GTA. RUIZ JIMENEZ / CALL. SAN BERNARDO,3,7,CHAMBERÍ,Colisión lateral,Despejado,Turismo,Conductor,De 25 a 29 años,Hombre,NaN,NaN,440137.0,4475721.0,N,NaN


## Column Analysis

Let's look at the available columns and some sample values for each.

In [2]:
analysis = []
for col in df.columns:
    analysis.append({
        "Column": col,
        "Unique Values": df[col].nunique(),
        "Sample Values": ", ".join(map(str, df[col].dropna().unique()[:5])),
        "Nulls": df[col].isna().sum()
    })

pd.DataFrame(analysis)

,Column,Unique Values,Sample Values,Nulls
0,num_expediente,20698,"2023S040280, 2023S040309, 2023S040310, 2023S04...",0
1,fecha,366,"04/01/2024, 15/02/2024, 18/02/2024, 25/02/2024...",0
2,hora,1256,"14:09:00, 14:05:00, 10:40:00, 17:20:00, 14:25:00",0
3,localizacion,15131,"AVDA. NICETO ALCALA ZAMORA / AUTOV. M-11, CALL...",0
4,numero,1759,"3, 18, 93, 14, 0",0
5,cod_distrito,21,"16, 1, 7, 6, 10",0
6,distrito,21,"HORTALEZA, CENTRO, CHAMBERÍ, TETUÁN, LATINA",0
7,tipo_accidente,12,"Colisión fronto-lateral, Colisión lateral, Alc...",5
8,estado_meteorológico,7,"Lluvia débil, Despejado, Nublado, Se desconoce...",5916
9,tipo_vehiculo,32,"Motocicleta > 125cc, Turismo, Bicicleta, Motoc...",394


In [7]:
for col in df.columns:
    if df[col].nunique() < 35:
        print(f"Column: {col}")
        print(f"Values: {', '.join(map(str, df[col].dropna().unique()))}")
        print(f"Nulls: {df[col].isna().sum()}")
        print("-" * 30)




Column: cod_distrito
Values: 16, 1, 7, 6, 10, 9, 3, 13, 14, 11, 4, 5, 19, 2, 8, 12, 18, 21, 17, 20, 15
Nulls: 0
------------------------------
Column: distrito
Values: HORTALEZA, CENTRO, CHAMBERÍ, TETUÁN, LATINA, MONCLOA-ARAVACA, RETIRO, PUENTE DE VALLECAS, MORATALAZ, CARABANCHEL, SALAMANCA, CHAMARTÍN, VICÁLVARO, ARGANZUELA, FUENCARRAL-EL PARDO, USERA, VILLA DE VALLECAS, BARAJAS, VILLAVERDE, SAN BLAS-CANILLEJAS, CIUDAD LINEAL
Nulls: 0
------------------------------
Column: tipo_accidente
Values: Colisión fronto-lateral, Colisión lateral, Alcance, Choque contra obstáculo fijo, Colisión múltiple, Colisión frontal, Atropello a persona, Caída, Solo salida de la vía, Otro, Vuelco, Atropello a animal
Nulls: 5
------------------------------
Column: estado_meteorológico
Values: Lluvia débil, Despejado, Nublado, Se desconoce, LLuvia intensa, Granizando, Nevando
Nulls: 5916
------------------------------
Column: tipo_vehiculo
Values: Motocicleta > 125cc, Turismo, Bicicleta, Motocicleta hasta 125

## Typical participant-level columns

Based on the data, these columns vary per person for the same `num_expediente`:

In [3]:
participant_cols = [
    "tipo_persona",
    "rango_edad",
    "sexo",
    "cod_lesividad",
    "lesividad",
    "tipo_vehiculo"
]

# Show an example of multiple participants in one accident
multi_participant_ids = df['num_expediente'].value_counts()
example_id = multi_participant_ids[multi_participant_ids > 1].index[0]

print(f"Example accident ID with multiple participants: {example_id}")
df[df['num_expediente'] == example_id][participant_cols]

Example accident ID with multiple participants: 2024S026225


,tipo_persona,rango_edad,sexo,cod_lesividad,lesividad,tipo_vehiculo
33369,Conductor,De 25 a 29 años,Hombre,6.0,Asistencia sanitaria inmediata en centro de sa...,Sin especificar
33370,Conductor,Desconocido,Desconocido,NaN,NaN,Sin especificar
33371,Conductor,Desconocido,Desconocido,NaN,NaN,Sin especificar
33372,Conductor,Desconocido,Desconocido,NaN,NaN,Sin especificar
33373,Pasajero,De 30 a 34 años,Hombre,5.0,Asistencia sanitaria ambulatoria con posterior...,Sin especificar
33374,Conductor,Desconocido,Desconocido,NaN,NaN,Todo terreno
33375,Conductor,De 25 a 29 años,Hombre,14.0,Sin asistencia sanitaria,Turismo
33376,Conductor,De 35 a 39 años,Hombre,5.0,Asistencia sanitaria ambulatoria con posterior...,Turismo
33377,Conductor,De 35 a 39 años,Hombre,6.0,Asistencia sanitaria inmediata en centro de sa...,Turismo
33378,Conductor,De 40 a 44 años,Hombre,3.0,Ingreso superior a 24 horas,Turismo
